## GOLD MASTER: VERDAD HISTORICA DE COLOMBIA (2019-2026)

In [3]:
## ============================================================================
## 🏆 GOLD MASTER: SERIE HISTÓRICA REFINADA (2019-2023)
## ============================================================================
from pyspark.sql import functions as F

gold_table = "dane_gold_lh.fact_labor_market"
years = [2019, 2020, 2021, 2022, 2023,2024,2025,2026]
dfs_to_union = []

print("🔗 Unificando las 8 tablas Silver...")

for y in years:
    # Cargamos cada tabla que ya limpiamos año por año
    df_s = spark.table(f"dane_silver_lh.labor_{y}")
    
    # Lógica de geografía para asegurar el Total Nacional
    df_geo = df_s.withColumn("is_total", F.when(F.col("is_national_total") == True, 1).otherwise(0)) \
        .groupBy("year", "month", "status").agg(
            F.sum(F.when(F.col("is_total") == 1, F.col("weight"))).alias("sum_nacional"),
            F.sum(F.when(F.col("geo_source") == "area", F.col("weight"))).alias("sum_area")
        ).withColumn("weight_final", F.when(F.col("sum_nacional") > 0, F.col("sum_nacional")).otherwise(F.col("sum_area")))
    
    dfs_to_union.append(df_geo.select("year", "month", "status", "weight_final"))

# 1. Unión de la serie completa
df_gold_raw = dfs_to_union[0]
for d in dfs_to_union[1:]:
    df_gold_raw = df_gold_raw.unionByName(d)

# 2. Pivote (Aquí es donde se crean las columnas 'desocupado' y 'ocupado')
df_fact = df_gold_raw.groupBy("year", "month").pivot("status").agg(F.sum("weight_final"))

# 3. Cálculos Finales (Usando nombres en singular como pidió Spark)
# PEA = Población Económicamente Activa
df_fact = df_fact.withColumn("pea", F.col("ocupado") + F.col("desocupado")) \
                 .withColumn("tasa_desempleo", F.col("desocupado") / F.col("pea"))

# 4. Guardar en Gold
df_fact.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_table)

print(f"✅ TABLA GOLD GENERADA: {gold_table}")
print("\n📈 SERIE HISTÓRICA FINAL:")
df_fact.orderBy("year", "month").show(100)

from pyspark.sql import functions as F

# 1. Cargamos la tabla que ya tienes
df_fact = spark.table("dane_gold_lh.fact_labor_market")

# 2. Creamos una columna 'date' real combinando año y mes (ej: 2023-01-01)
df_fact = df_fact.withColumn("date", 
    F.to_date(F.concat(F.col("year"), F.lit("-"), F.col("month"), F.lit("-01")))
)

# 3. Guardamos de nuevo (sobrescribiendo)
df_fact.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dane_gold_lh.fact_labor_market")

print("✅ Columna de fecha inyectada. ¡Ahora la relación será fácil!")

StatementMeta(, 0c9a62e5-bf4c-470c-ac1f-31643ace6c1c, 5, Finished, Available, Finished, False)

🔗 Unificando las 8 tablas Silver...
✅ TABLA GOLD GENERADA: dane_gold_lh.fact_labor_market

📈 SERIE HISTÓRICA FINAL:
+----+-----+-------------------+--------------------+--------------------+--------------------+
|year|month|         desocupado|             ocupado|                 pea|      tasa_desempleo|
+----+-----+-------------------+--------------------+--------------------+--------------------+
|2019|    1|  3176668.441605063| 2.164986065292175E7|2.4826529094526812E7|  0.1279545936328805|
|2019|    2| 2943871.4547956567| 2.207118179397165E7| 2.501505324876731E7| 0.11768399713243563|
|2019|    3| 2682092.1622664644|2.2114054891322784E7|2.4796147053589247E7| 0.10816568221143176|
|2019|    4| 2523608.9674953762| 2.189602840752081E7|2.4419637375016183E7| 0.10334342515983835|
|2019|    5| 2610690.8382819803| 2.216421358019582E7|2.4774904418477803E7| 0.10537642423091874|
|2019|    6|  2356914.354858337| 2.261808440595074E7| 2.497499876080908E7| 0.09437094982190035|
|2019|    7| 2657297

In [4]:
# 🧹 PULIDO FINAL: ELIMINACIÓN DE OUTLIERS EXTREMOS
from pyspark.sql import functions as F

df_gold = spark.table("dane_gold_lh.fact_labor_market")

# Filtramos valores que sean astronómicamente imposibles 
# (Colombia no tiene más de 55M de habitantes)
poblacion_maxima = 60000000 

df_gold_clean = df_gold.withColumn(
    "ocupado", 
    F.when(F.col("ocupado") > poblacion_maxima, F.lit(None)).otherwise(F.col("ocupado"))
).withColumn(
    "desocupado", 
    F.when(F.col("desocupado") > poblacion_maxima, F.lit(None)).otherwise(F.col("desocupado"))
)

# Recalculamos la tasa para los valores limpios
df_gold_clean = df_gold_clean.withColumn("pea", F.col("ocupado") + F.col("desocupado")) \
                             .withColumn("tasa_desempleo", F.col("desocupado") / F.col("pea"))

# Guardamos la versión final "Espejo"
df_gold_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dane_gold_lh.fact_labor_market")

print("✨ Tabla Gold pulida. ¡Outliers espaciales eliminados!")

StatementMeta(, 0c9a62e5-bf4c-470c-ac1f-31643ace6c1c, 6, Finished, Available, Finished, False)

✨ Tabla Gold pulida. ¡Outliers espaciales eliminados!


## DIMDATE TABLE

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# 1. Configuración de rango
start_date = "2019-01-01"

# 2. Generar secuencia de fechas hasta hoy
df_dates = spark.createDataFrame([(start_date,)], ["start"]) \
    .select(F.explode(F.sequence(F.to_date("start"), F.current_date(), F.expr("interval 1 day"))).alias("date"))

# 3. Extraer atributos de tiempo
dim_date = df_dates \
    .withColumn("date_key",    F.date_format("date", "yyyyMMdd").cast("int")) \
    .withColumn("year",        F.year("date")) \
    .withColumn("year_text",   F.year("date").cast("string")) \
    .withColumn("quarter",     F.quarter("date")) \
    .withColumn("month",       F.month("date")) \
    .withColumn("month_name",  F.date_format("date", "MMMM")) \
    .withColumn("month_short", F.date_format("date", "MMM")) \
    .withColumn("day",         F.dayofmonth("date")) \
    .withColumn("day_of_week", F.dayofweek("date")) \
    .withColumn("day_name",    F.date_format("date", "EEEE")) \
    .withColumn("week_of_year",F.weekofyear("date")) \
    .withColumn("is_weekend",  F.when(F.col("day_of_week").isin(1, 7), True).otherwise(False))

# 4. Guardar en Gold — Añadimos la opción para forzar el nuevo esquema
dim_date.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dane_gold_lh.dim_date")

print("✅ DIM_DATE actualizada. El esquema viejo fue reemplazado por el nuevo con éxito.")

StatementMeta(, 5f3a6054-b64f-429d-86c7-97b81cc3cca8, 6, Finished, Available, Finished, False)

✅ DIM_DATE actualizada. El esquema viejo fue reemplazado por el nuevo con éxito.


In [6]:
%%sql
SELECT * FROM dane_gold_lh.fact_labor_market

StatementMeta(, 0c9a62e5-bf4c-470c-ac1f-31643ace6c1c, 8, Finished, Available, Finished, False)

<Spark SQL result set with 86 rows and 8 fields>

In [5]:
from pyspark.sql import functions as F

# Aplicar la lógica de periodos presidenciales desde 2002
df_gold = df_gold.withColumn("presidente", 
    F.when((F.col("date") >= "2002-08-07") & (F.col("date") < "2010-08-07"), "Álvaro Uribe")
     .when((F.col("date") >= "2010-08-07") & (F.col("date") < "2018-08-07"), "Juan Manuel Santos")
     .when((F.col("date") >= "2018-08-07") & (F.col("date") < "2022-08-07"), "Iván Duque")
     .when(F.col("date") >= "2022-08-07", "Gustavo Petro")
     .otherwise("Periodo Anterior")
)

# Guardar con overwriteSchema para que Power BI vea la nueva columna
df_gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dane_gold_lh.fact_labor_market")

print("✅ Columna 'presidente' inyectada con éxito en la tabla Gold.")

StatementMeta(, 0c9a62e5-bf4c-470c-ac1f-31643ace6c1c, 7, Finished, Available, Finished, False)

✅ Columna 'presidente' inyectada con éxito en la tabla Gold.
